# Horizon Evaluation Report

This notebook evaluates the experiments separately for:

- daily prediction (`horizon = 1`)
- weekly prediction (`horizon = 5`)
- monthly prediction (`horizon = 21`)

For each horizon it:

1. ranks the candidate runs,
2. selects the best model,
3. shows the evidence for why it was chosen,
4. surfaces robustness and multimodal lift,
5. gives presentation-ready reasoning.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from stock_forecasting.reporting import (
    build_prediction_diagnostics,
    discover_run_dirs,
    horizon_label,
    load_predictions_for_run,
    load_results,
    rank_runs,
    summarize_horizon,
)

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="talk")
    HAVE_SEABORN = True
except ImportError:
    HAVE_SEABORN = False
    plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_ROOT = Path("artifacts")
SELECTED_TASK = "regression"
TOP_N = 8
HORIZONS = [1, 5, 21]

pd.options.display.max_columns = 200
pd.options.display.float_format = lambda x: f"{x:,.4f}"

from stock_forecasting.visualization import (
    plot_random_stock_candlestick_forecasts,
    plot_random_stock_history_forecasts,
)


In [ ]:
def modality_lift_for_horizon(horizon_df, metric_name):
    needed = {"model_name", "modalities", metric_name}
    if horizon_df.empty or not needed.issubset(horizon_df.columns):
        return pd.DataFrame()
    pivot = horizon_df.pivot_table(
        index="model_name",
        columns="modalities",
        values=metric_name,
        aggfunc="mean",
    )
    if {"price", "price_news"}.issubset(pivot.columns):
        pivot = pivot.copy()
        pivot["lift_price_news_minus_price"] = pivot["price_news"] - pivot["price"]
        pivot = pivot.sort_values("lift_price_news_minus_price", ascending=False)
        return pivot.reset_index()
    return pd.DataFrame()


def safe_value(series, key):
    value = series.get(key, np.nan)
    return value if pd.notna(value) else np.nan


def display_horizon_report(horizon, ranked_df, folds_df, main_metric):
    horizon_df = ranked_df.loc[ranked_df["horizon"] == horizon].copy()
    if horizon_df.empty:
        display(Markdown(f"## {horizon_label(horizon)}\nNo completed runs found for this horizon."))
        return None

    summary = summarize_horizon(ranked_df, horizon, main_metric)
    best = summary["best"]
    runner_up = summary["runner_up"]
    title = horizon_label(horizon)

    evidence_lines = [
        f"## {title}",
        "",
        f"Selected model: `{best['run_name']}`",
        "",
        f"- Model family: `{best['model_name']}`",
        f"- Inputs: `{best['modalities']}`",
        f"- Lookback: `{best['lookback']}` trading days",
        f"- News lag: `{best.get('news_lag_days', 'n/a')}` day(s)",
        f"- Main metric `{main_metric}`: `{safe_value(best, main_metric):.4f}`",
        f"- Composite score: `{safe_value(best, 'composite_score'):.4f}`",
    ]

    if pd.notna(safe_value(best, f"{main_metric}__std")):
        evidence_lines.append(
            f"- Fold stability: `{safe_value(best, f'{main_metric}__std'):.4f}` std on `{main_metric}`"
        )
    if pd.notna(safe_value(best, "test_directional_accuracy")):
        evidence_lines.append(
            f"- Directional accuracy: `{safe_value(best, 'test_directional_accuracy'):.2%}`"
        )
    if pd.notna(safe_value(best, "test_top_bottom_decile_spread")):
        evidence_lines.append(
            f"- Top-bottom decile spread: `{safe_value(best, 'test_top_bottom_decile_spread'):.4f}`"
        )
    if runner_up is not None and pd.notna(safe_value(best, main_metric)) and pd.notna(safe_value(runner_up, main_metric)):
        evidence_lines.append(
            f"- Gap over runner-up on `{main_metric}`: `{safe_value(best, main_metric) - safe_value(runner_up, main_metric):.4f}`"
        )

    display(Markdown("\n".join(evidence_lines)))

    leaderboard_columns = [
        "run_name", "model_name", "modalities", "lookback", "news_lag_days", "seed",
        "fold_count", "composite_score", main_metric,
    ]
    optional_columns = [
        f"{main_metric}__std", "test_directional_accuracy", "test_top_bottom_decile_spread",
        "test_rmse", "test_mae", "test_spearman_ic",
    ]
    leaderboard_columns.extend([column for column in optional_columns if column in horizon_df.columns])
    display(Markdown("### Horizon leaderboard"))
    display(horizon_df[leaderboard_columns].head(TOP_N))

    horizon_folds = folds_df.loc[folds_df["horizon"] == horizon].copy() if not folds_df.empty else pd.DataFrame()
    if not horizon_folds.empty and main_metric in horizon_folds.columns:
        top_names = horizon_df.head(min(5, len(horizon_df)))["run_name"].tolist()
        fold_plot_df = horizon_folds.loc[horizon_folds["run_name"].isin(top_names)]
        plt.figure(figsize=(14, 5))
        if HAVE_SEABORN:
            sns.boxplot(data=fold_plot_df, x=main_metric, y="run_name", color="#a6cee3")
            sns.stripplot(data=fold_plot_df, x=main_metric, y="run_name", color="#1f78b4", size=5, alpha=0.85)
        else:
            grouped = [fold_plot_df.loc[fold_plot_df["run_name"] == name, main_metric].dropna().values for name in top_names]
            plt.boxplot(grouped, vert=False, labels=top_names)
        plt.title(f"{title}: fold robustness on {main_metric}")
        plt.xlabel(main_metric)
        plt.ylabel("Run")
        plt.tight_layout()
        plt.show()

    lift_df = modality_lift_for_horizon(horizon_df, main_metric)
    if not lift_df.empty:
        display(Markdown("### News and sentiment lift"))
        display(lift_df)
        plt.figure(figsize=(10, 4))
        plt.barh(lift_df["model_name"], lift_df["lift_price_news_minus_price"], color="#33a02c")
        plt.axvline(0.0, color="black", linestyle="--", linewidth=1)
        plt.title(f"{title}: multimodal lift on {main_metric}")
        plt.xlabel("price_news - price")
        plt.tight_layout()
        plt.show()

    predictions_df = load_predictions_for_run(best["run_name"], ranked_df)
    daily_df, ticker_df, magnitude_df = build_prediction_diagnostics(predictions_df)
    if not predictions_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(18, 5))
        sample_df = predictions_df.sample(min(3000, len(predictions_df)), random_state=7)
        axes[0].scatter(sample_df["prediction"], sample_df["target"], alpha=0.20, color="#1f78b4")
        axes[0].axhline(0.0, color="black", linewidth=1, linestyle="--")
        axes[0].axvline(0.0, color="black", linewidth=1, linestyle="--")
        axes[0].set_title(f"{title}: prediction vs actual")
        axes[0].set_xlabel("Prediction")
        axes[0].set_ylabel("Target")

        if not daily_df.empty:
            axes[1].plot(daily_df["date"], daily_df["cumulative_spread"], color="#33a02c", linewidth=2)
            axes[1].axhline(0.0, color="black", linewidth=1, linestyle="--")
            axes[1].set_title(f"{title}: cumulative top-bottom spread")
            axes[1].set_xlabel("Date")
            axes[1].set_ylabel("Cumulative spread")
        else:
            axes[1].text(0.5, 0.5, "Not enough cross-sectional samples", ha="center", va="center")
            axes[1].set_axis_off()

        plt.tight_layout()
        plt.show()

        if not daily_df.empty:
            plt.figure(figsize=(14, 4))
            plt.plot(daily_df["date"], daily_df["daily_ic"], color="#6a3d9a", alpha=0.35, label="Daily IC")
            plt.plot(daily_df["date"], daily_df["rolling_ic_20"], color="#e31a1c", linewidth=2, label="20-day rolling IC")
            plt.axhline(0.0, color="black", linewidth=1, linestyle="--")
            plt.title(f"{title}: rank quality over time")
            plt.xlabel("Date")
            plt.ylabel("Spearman IC")
            plt.legend()
            plt.tight_layout()
            plt.show()

        display(Markdown("### Most reliable tickers for the selected model"))
        display(ticker_df.head(15))

        if not magnitude_df.empty:
            display(Markdown("### Error behavior by target magnitude"))
            display(magnitude_df)

        try:
            fig, selected_tickers = plot_random_stock_history_forecasts(
                predictions_df,
                price_csv=best.get("price_csv", "data/dates_on_left_stock_data.csv"),
                horizon=int(horizon),
                title=f"{title}: stock history with actual test prices vs predicted prices for random tickers",
                n_tickers=5,
                seed=7,
            )
            display(Markdown("### Stock-history test vs prediction charts"))
            display(Markdown(f"Randomly selected tickers: `{", ".join(selected_tickers)}`"))
            fig.show()
        except Exception as exc:
            display(Markdown(f"_Stock-history chart unavailable: {exc}_"))

    why_lines = [
        f"### Why this {title.lower()} model was chosen",
        "",
        f"- It achieved the highest composite score among `{len(horizon_df)}` candidate runs for this horizon.",
        f"- It led on the main selection metric `{main_metric}` or was very close while remaining more stable across folds.",
        f"- It uses `{best['modalities']}` inputs, which lets us verify whether news and sentiment added measurable value.",
        f"- Its lookback of `{best['lookback']}` and news lag of `{best.get('news_lag_days', 'n/a')}` are directly visible, making the choice defensible in a presentation.",
    ]
    display(Markdown("\n".join(why_lines)))
    return summary


In [ ]:
run_dirs = discover_run_dirs(RESULTS_ROOT)
runs_df, folds_df = load_results(run_dirs)

if runs_df.empty:
    display(Markdown("## No completed runs found\nRun the training suite first, then reopen this notebook."))
else:
    if SELECTED_TASK is not None and "task" in runs_df.columns:
        analysis_df = runs_df.loc[runs_df["task"] == SELECTED_TASK].copy()
        analysis_folds_df = folds_df.loc[folds_df["task"] == SELECTED_TASK].copy() if not folds_df.empty else folds_df.copy()
    else:
        analysis_df = runs_df.copy()
        analysis_folds_df = folds_df.copy()

    ranked_df, MAIN_METRIC = rank_runs(analysis_df)
    display(Markdown(f"## Loaded {len(ranked_df)} runs for task `{SELECTED_TASK}`"))
    display(ranked_df[["run_name", "model_name", "modalities", "horizon", "lookback", "news_lag_days", "composite_score", MAIN_METRIC]].head(12))


## Daily, Weekly, and Monthly Model Selection

Each section below makes an independent recommendation for that horizon, along with the evidence needed to defend the choice.

In [ ]:
horizon_summaries = []

if not runs_df.empty:
    for horizon in HORIZONS:
        summary = display_horizon_report(horizon, ranked_df, analysis_folds_df, MAIN_METRIC)
        if summary is not None:
            horizon_summaries.append(summary)


## Candlestick Overlay By Horizon

This cell gives a market-style chart for each selected horizon winner, using close-derived candlesticks from the historical series and overlaying actual test prices and predicted prices.


In [ ]:
if horizon_summaries:
    for summary in horizon_summaries:
        best = summary["best"]
        predictions_df = load_predictions_for_run(best["run_name"], ranked_df)
        if predictions_df.empty:
            continue
        try:
            fig, selected_tickers, note = plot_random_stock_candlestick_forecasts(
                predictions_df,
                price_csv=best.get("price_csv", "data/dates_on_left_stock_data.csv"),
                horizon=int(summary["horizon"]),
                title=f"{summary['label']}: close-derived candlesticks with actual vs predicted test prices",
                n_tickers=5,
                seed=17,
            )
            display(Markdown(f"### {summary['label']} candlestick-style chart"))
            display(Markdown(f"Randomly selected tickers: `{", ".join(selected_tickers)}`"))
            display(Markdown(f"_{note}_"))
            fig.show()
        except Exception as exc:
            display(Markdown(f"_Candlestick chart unavailable for {summary['label']}: {exc}_"))


## Final Recommendation Table

This table is the compact summary you would typically put into a slide or executive summary.

In [ ]:
if horizon_summaries:
    final_rows = []
    for summary in horizon_summaries:
        best = summary["best"]
        final_rows.append(
            {
                "report": summary["label"],
                "horizon": summary["horizon"],
                "selected_run": best["run_name"],
                "model_family": best["model_name"],
                "modalities": best["modalities"],
                "lookback": best["lookback"],
                "news_lag_days": best.get("news_lag_days", np.nan),
                "main_metric": summary["main_metric"],
                "main_metric_value": summary["main_metric_value"],
                "fold_std": summary["stability"],
                "directional_accuracy": summary["directional_accuracy"],
                "top_bottom_spread": summary["spread"],
                "gap_to_runner_up": summary["metric_gap_to_runner_up"],
            }
        )

    final_df = pd.DataFrame(final_rows).sort_values("horizon")
    display(final_df)

    takeaway_lines = [
        "## Competition-ready summary",
        "",
    ]
    for _, row in final_df.iterrows():
        takeaway_lines.append(
            f"- {row['report']}: choose `{row['selected_run']}` because it led on `{row['main_metric']}` at `{row['main_metric_value']:.4f}`, with fold std `{row['fold_std']:.4f}` and directional accuracy `{row['directional_accuracy']:.2%}`."
        )
    display(Markdown("\n".join(takeaway_lines)))
